# 10 · ปรับ KNN ด้วย GridSearchCV และอ่าน learning curve

[กลับสารบัญ](../../README.md) · [Notebook](lesson.ipynb) · [Python](lesson.py)

บทนี้ใช้ **GridSearchCV** เลือก hyperparameter และอ่าน learning curve เพื่อดูผลของขนาดข้อมูลฝึก

**ใช้สิ่งที่เรียนแล้ว:** KNN/Pipeline จากบท 03, CV จากบท 08 และ overfitting จากบท 09

## โมเดลที่ปรับ: KNN จากบท 03

**KNeighborsClassifier** รับขนาดกลีบสี่ค่าหลังปรับสเกล แล้วทายสายพันธุ์จากคำตอบของเพื่อนบ้านใกล้ที่สุด ทบทวนหลักการใน [บท 03](../03-first-classifier/README.md)

บทนี้ลองจำนวนเพื่อนบ้านหลายค่า และเปรียบเทียบการลงคะแนนเท่ากันกับการให้จุดใกล้มีน้ำหนักมากกว่า เพื่อเลือกวิธีที่เหมาะกับข้อมูลฝึก **GridSearchCV เป็นเครื่องมือเลือกการตั้งค่า ส่วน KNN ยังเป็นตัวทำนาย** ผลเลือกอาจเปลี่ยนตามข้อมูลและ metric จึงต้องเก็บ test ไว้ตรวจท้ายสุด

![ทบทวน KNN เมื่อ k เท่ากับ 5 และให้น้ำหนักเสียงเท่ากัน เพื่อนบ้าน versicolor 3 จุดชนะ virginica 2 จุด](../../assets/figures/03-first-classifier-knn.png)

**ทบทวนภาพจากบท 03:** ภาพใช้ `k=5, weights="uniform"` บทนี้จะลองเปลี่ยนทั้ง k และวิธีให้น้ำหนัก; เมื่อใช้ `distance` จุดใกล้มีน้ำหนักมากกว่า จึงไม่ได้ตัดสินจากจำนวนเสียงอย่างเดียว · [เปิดภาพ SVG](../../assets/figures/03-first-classifier-knn.svg)

## ใจความใหม่

- **Parameter:** ค่าที่เรียนจากข้อมูล เช่น coefficient
- **Hyperparameter:** การตั้งค่าวิธีเรียน เช่น k ของ KNN

### GridSearchCV คืออะไร

**Grid Search with Cross-validation** คือการค้นหาค่า hyperparameter โดยลองทุกชุดค่าที่กำหนด และประเมินแต่ละชุดด้วย CV เครื่องมือนี้ช่วยเลือกการตั้งค่าให้โมเดล เช่น KNN ที่เรียนแล้ว

**หลักการ:** กำหนดรายการค่าที่จะลอง → ฝึกและวัดผลแต่ละชุดใน fold เดียวกัน → เปรียบเทียบคะแนนเฉลี่ย → เลือกชุดที่ดีที่สุดในรายการ

ตัวอย่าง: k=3, 5, 9 × weights สองแบบ = 6 ชุด หากใช้ 5-fold CV จะฝึก 30 ครั้ง แล้วฝึกชุดที่ชนะบน train ทั้งหมดอีกครั้ง เรียกว่า **refit**

- **จุดเด่น:** เปรียบเทียบชุดค่าตามกระบวนการเดียวกัน และตรวจย้อนกลับได้ว่าแต่ละชุดได้คะแนนเท่าไร
- **ข้อจำกัด:** จำนวนการฝึกเพิ่มตามจำนวนชุดค่าและ fold อีกทั้งคำว่า “ดีที่สุด” หมายถึงดีที่สุดในรายการที่ลอง ตามข้อมูลและ metric ที่ใช้เท่านั้น

Scaler อยู่ใน Pipeline ตลอดการค้นหา

คะแนน best CV อาจสูงเกินจริง เพราะใช้คะแนนนี้เลือกค่าไปแล้ว จึงต้องประเมิน test หลังเลือกเสร็จ

## อ่าน learning curve

**Learning curve — เส้นโค้งการเรียนรู้** แสดงคะแนน train และ validation เมื่อเปลี่ยนจำนวนตัวอย่างฝึก ใช้ตรวจว่าข้อมูลเพิ่มอาจช่วยหรือควรปรับโมเดล โดยฝึกใหม่ที่แต่ละขนาดข้อมูล กราฟนี้จึงต่างจากกราฟ loss ตามจำนวนรอบฝึกในบท 15

| รูปแบบคะแนนเมื่อเพิ่มข้อมูลฝึก | สิ่งที่ควรตรวจ |
|---|---|
| train และ validation ต่ำใกล้กัน | feature หรือความสามารถโมเดลยังไม่พอ |
| train สูง แต่ validation ต่ำกว่า | อาจ overfit ให้ลองจำกัดความซับซ้อนหรือเพิ่มข้อมูล |
| validation สูงขึ้น | ข้อมูลเพิ่มอาจช่วย แต่ไม่รับประกันว่าจะดีขึ้นเสมอ |

**แกนนอน:** จำนวนตัวอย่างที่ใช้ฝึกในแต่ละ fold ส่วนแกนตั้งคือ accuracy บน train และ validation ของการฝึกขนาดนั้น

กราฟนี้ใช้ค่าที่เลือกจาก train เดิม จึงเป็นเครื่องมือวินิจฉัย ไม่ใช่ผลประเมินอิสระอีกชุด

## ลงมือทำ

สร้าง Notebook ใหม่แล้วรัน **Cell 1–2 ตามลำดับ** หรือเปิด [Notebook ของบทนี้](lesson.ipynb) ใช้ kernel เดิมได้เลย อ่านคำอธิบายใต้โค้ดแต่ละช่วงประกอบผลที่ได้

<!-- CODE:CELLS -->



### Cell 1 — ค้นค่าที่เหมาะด้วย GridSearchCV


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, learning_curve, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
pipeline = Pipeline([("scale", StandardScaler()), ("knn", KNeighborsClassifier())])
search = GridSearchCV(
    pipeline, {"knn__n_neighbors": [3, 5, 9], "knn__weights": ["uniform", "distance"]},
    scoring="accuracy", cv=cv, n_jobs=1,
)
search.fit(X_train, y_train)
results = pd.DataFrame(search.cv_results_)
print(results[["param_knn__n_neighbors", "param_knn__weights",
               "mean_test_score", "std_test_score", "rank_test_score"]]
      .sort_values("rank_test_score").round(3).to_string(index=False))
print("Best parameters:", search.best_params_)
print(f"Best CV accuracy: {search.best_score_:.3f}")



**อ่านโค้ด**

- `load_iris(return_X_y=True)` คืน X และ y โดยตรง แทนการคืนชุดข้อมูลพร้อมรายละเอียดแบบบท 02
- `Pipeline([("scale", ...), ("knn", ...)])` ตั้งชื่อแต่ละขั้นเอง เพื่ออ้างชื่อในการค้นพารามิเตอร์
- `knn__n_neighbors` และ `knn__weights` ใช้รูปแบบ `ชื่อขั้น__ชื่อพารามิเตอร์` ให้ GridSearchCV รู้ว่าต้องปรับค่าของ KNN
- `search.fit(X_train, y_train)` ทดลองหกชุดค่าด้วยห้า fold แล้ว refit ชุดที่ชนะบน train ทั้งหมดตามค่าเริ่มต้น
- `pd.DataFrame(search.cv_results_)` ทำผลค้นหาเป็นตาราง แล้วเรียงด้วย `rank_test_score` เพื่ออ่านลำดับจากคะแนน CV
- `best_params_` บอกชุดค่าที่เลือก และ `best_score_` เป็นคะแนน CV เฉลี่ยของชุดนั้น ไม่ใช่ final test

**ชื่อที่ใช้ในตารางผล:** `mean_test_score` คือ accuracy เฉลี่ยของ validation ใน CV และ `std_test_score` คือส่วนเบี่ยงเบนมาตรฐานระหว่าง fold ไม่ใช่คะแนน final test

`weights="uniform"` ให้เพื่อนบ้านลงคะแนนเท่ากัน ส่วน `"distance"` ให้จุดใกล้มีน้ำหนักมากกว่า



### Cell 2 — ดูผลของจำนวนข้อมูลฝึก


In [ ]:
sizes, train_scores, valid_scores = learning_curve(
    search.best_estimator_, X_train, y_train, cv=cv, scoring="accuracy",
    train_sizes=[0.4, 0.6, 0.8, 1.0], shuffle=True, random_state=42,
)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sizes, train_scores.mean(axis=1), "o-", label="Train")
ax.plot(sizes, valid_scores.mean(axis=1), "o-", label="Validation")
ax.set(xlabel="Training examples per fold", ylabel="Accuracy", ylim=(0, 1.05))
ax.legend()
fig.tight_layout()
out = Path("outputs/10-tuning-and-learning-curves")
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "learning-curve.png", dpi=140)
plt.show()
plt.close(fig)
print(f"Final test accuracy: {accuracy_score(y_test, search.predict(X_test)):.3f}")



**อ่านโค้ด**

- `learning_curve(search.best_estimator_, ...)` ใช้การตั้งค่าที่เลือกแล้ว ฝึกซ้ำด้วยขนาด train หลายระดับ
- `train_sizes=[0.4, 0.6, 0.8, 1.0]` เป็นสัดส่วนของข้อมูลฝึกที่ใช้ได้ในแต่ละ fold ระดับ 1.0 จึงเท่ากับ 96 แถวในบทนี้
- `sizes` คืนจำนวนแถวจริง ส่วน `train_scores` และ `valid_scores` มีหนึ่งแถวต่อขนาดข้อมูล และหนึ่งคอลัมน์ต่อ fold
- `mean(axis=1)` เฉลี่ยคะแนนข้าม fold เพื่อวาดหนึ่งจุดต่อขนาดข้อมูล
- `search.predict(X_test)` ใช้โมเดลที่ GridSearchCV refit ไว้ประเมิน test หลังเลือกค่าเสร็จแล้ว

**ที่มาของ 96 แถวบนแกนนอน:** train ทั้งหมดมี 120 แถว แต่แต่ละรอบของ 5-fold CV ใช้ฝึกเพียง 4 ส่วน จึงมีสูงสุด `120 × 4/5 = 96` แถว ส่วนอีก 24 แถวใช้ validation




## อ่านผล

อ่านผลตามลำดับ: ตารางหกทางเลือก → best parameters → learning curve → test accuracy

ไฟล์ภาพ: `outputs/10-tuning-and-learning-curves/learning-curve.png`

![ผลการทดลอง 10-tuning-and-learning-curves](../../assets/figures/10-tuning-and-learning-curves-learning-curve.png)

## แบบฝึกท้ายบท

**ลองคิด:** k ห้าค่า × weights สองค่า × CV สี่ fold ต้อง fit กี่ครั้ง?

<details>
<summary>ดูเฉลย</summary>

**เฉลย:** ฝึก 40 ครั้งสำหรับ CV และ refit อีก 1 ครั้ง รวม 41 ครั้ง

อย่าใช้ผล test เลือกว่าจะขยาย grid อย่างไรซ้ำ ๆ เพราะจะปรับโมเดลเข้าหาความบังเอิญใน test

</details>

## อ้างอิง

- [Handbook: Hyperparameters and Model Validation](https://jakevdp.github.io/PythonDataScienceHandbook/05.03-hyperparameters-and-model-validation.html)
- [Tuning hyperparameters](https://scikit-learn.org/1.7/modules/grid_search.html)
- [Validation and learning curves](https://scikit-learn.org/1.7/modules/learning_curve.html)
